# 02 · Results
Run the models first (`make baseline models hybrids`, or `uv run vgrec run --model <name>`). This notebook only reads `results/`; it works with whichever models have results.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vgrec.config import RESULTS_DIR
from vgrec.data import load_dataset, load_items
from vgrec.experiment import report

tables = report()
summary = tables['summary']
cols = ['users', 'precision@10', 'recall@10', 'ndcg@10', 'hit_rate@10', 'coverage',
        'personalization', 'fit_s', 'score_s']
test = summary[summary['stage'] == 'test'].set_index('model')[cols]
test.sort_values('ndcg@10', ascending=False).round(4)

## Validation vs. test (did tuning overfit the valid stage?)

In [ ]:
summary.pivot_table(index='model', columns='stage', values='ndcg@10').round(4)

## NDCG@10 by history length (the proposal's *limited histories* question)

In [ ]:
by_hist = tables['ndcg_by_history']
ax = by_hist.drop(columns='users').plot.bar(figsize=(10, 4), ylabel='test NDCG@10',
                                             title='Test NDCG@10 by history length')
ax.set_xticklabels([f'{b}\n(n={n:,})' for b, n in by_hist['users'].items()], rotation=0);

## Are the differences significant? (paired bootstrap over test users)

In [ ]:
tables['significance'].round(5)

## Hybrid weight sweeps (valid stage)

In [ ]:
for name in ('hybrid', 'hybrid_pop'):
    path = RESULTS_DIR / name / 'valid_grid.csv'
    if path.exists():
        grid = pd.read_csv(path)
        print(name)
        display(grid.pivot_table(index='alpha', columns=['beta', 'adaptive_k'], values='ndcg@10').round(4))

## Qualitative check: one user's history and each model's top-10

In [ ]:
ds = load_dataset()
titles = load_items()['title'].str.slice(0, 70)
models = [p.parent.name for p in sorted(RESULTS_DIR.glob('*/test_topk.npy'))]
per_user = pd.read_parquet(RESULTS_DIR / models[0] / 'test_per_user.parquet')
row = int(np.flatnonzero(per_user['n_history'].between(3, 6))[0])
user = int(per_user.loc[row, 'user_idx'])
mine = ds.interactions[ds.interactions['user_idx'] == user].sort_values('timestamp')
print('History:', *titles.iloc[mine.loc[mine['split'] != 'test', 'item_idx']], sep='\n  ')
print('Interacted with next (test):', *titles.iloc[mine.loc[mine['split'] == 'test', 'item_idx']], sep='\n  ')
pd.DataFrame({m: titles.iloc[np.load(RESULTS_DIR / m / 'test_topk.npy')[row]].tolist() for m in models})

## Takeaways
Summarize: best model and margin (with CI), where content helps (history buckets), cost vs. accuracy (fit_s / score_s), and personalization/coverage trade-offs.